# CXR lung field segmentation only

Ez a notebook kizárólag a **tüdőmező-szegmentáció tanítására** és a meglévő CXR osztályozó datasetből a következő variánsok exportálására szolgál:

- `lung_masks`
- `lung_masked`
- `lung_crop`

A classifier / transfer learning rész **nincs** benne. Az lesz a következő notebook.


In [ ]:
!git clone https://github.com/csambrus/CXR.git
%cd CXR
!pip install -r ./requirements.txt

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import (
    RAW_DIR,
    SEGMENTATION_DIR,
    SEGMENTATION_MODELS_DIR,
    LUNG_MASK_DIR,
    LUNG_MASKED_DIR,
    LUNG_CROP_DIR,
    setup_tensorflow_runtime,
    ensure_dir,
    save_json,
)
from src.lung_segmentation import (
    discover_segmentation_pairs,
    split_pairs,
    make_segmentation_dataset,
    train_segmentation_model,
    evaluate_segmentation_model,
    save_metrics,
    plot_history,
    show_samples,
    show_predictions,
    export_masks_for_classification_dataset,
)

setup_tensorflow_runtime()

print("RAW_DIR:", RAW_DIR)
print("SEGMENTATION_DIR:", SEGMENTATION_DIR)

## Szegmentációs adatok megadása

Adj meg legalább egy olyan adatforrást, ahol kézzel annotált tüdőmaszkok vannak.

Támogatott elrendezések:

- **Montgomery**
  - `CXR_png/`
  - `ManualMask/leftMask/`
  - `ManualMask/rightMask/`
- **általános**
  - `images/`
  - `masks/`


In [ ]:
SEG_DATASET_ROOTS = {
    # "montgomery": "/content/drive/MyDrive/datasets/Montgomery",
    # "jsrt": "/content/drive/MyDrive/datasets/JSRT",
    # "shenzhen": "/content/drive/MyDrive/datasets/Shenzhen",
}

print(json.dumps(SEG_DATASET_ROOTS, indent=2, ensure_ascii=False))

In [ ]:
pairs = discover_segmentation_pairs(SEG_DATASET_ROOTS)
print("Discovered segmentation pairs:", len(pairs))

if len(pairs) > 0:
    show_samples(pairs, image_size=(256, 256), n=min(4, len(pairs)))
    plt.show()

In [ ]:
train_pairs, val_pairs, test_pairs = split_pairs(
    pairs,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
)

print("train:", len(train_pairs))
print("val  :", len(val_pairs))
print("test :", len(test_pairs))

In [ ]:
SEG_IMAGE_SIZE = (256, 256)
SEG_BATCH_SIZE = 8

train_ds = make_segmentation_dataset(
    train_pairs,
    image_size=SEG_IMAGE_SIZE,
    batch_size=SEG_BATCH_SIZE,
    training=True,
)
val_ds = make_segmentation_dataset(
    val_pairs,
    image_size=SEG_IMAGE_SIZE,
    batch_size=SEG_BATCH_SIZE,
    training=False,
)
test_ds = make_segmentation_dataset(
    test_pairs,
    image_size=SEG_IMAGE_SIZE,
    batch_size=SEG_BATCH_SIZE,
    training=False,
)

train_ds, val_ds, test_ds

In [ ]:
seg_model_dir = ensure_dir(SEGMENTATION_MODELS_DIR / "unet_lung_fields")

model, history = train_segmentation_model(
    train_ds=train_ds,
    val_ds=val_ds,
    model_dir=seg_model_dir,
    input_shape=(SEG_IMAGE_SIZE[0], SEG_IMAGE_SIZE[1], 1),
    base_filters=32,
    epochs=25,
)

plot_history(history, save_path=seg_model_dir / "history.png")
plt.show()

In [ ]:
metrics = evaluate_segmentation_model(model, test_ds)
metrics

In [ ]:
save_metrics(metrics, seg_model_dir / "metrics.json")
print("Saved:", seg_model_dir / "metrics.json")

In [ ]:
if len(test_pairs) > 0:
    show_predictions(
        model=model,
        pairs=test_pairs,
        image_size=SEG_IMAGE_SIZE,
        n=min(4, len(test_pairs)),
    )
    plt.show()

## Export a classifier projekthez

Ez a lépés a `RAW_DIR` alatt lévő teljes CXR osztályozó datasetre lefuttatja a tanult szegmentációs modellt, és három kimenetet készít:

- `LUNG_MASK_DIR`
- `LUNG_MASKED_DIR`
- `LUNG_CROP_DIR`

A relatív mappastruktúra megmarad, így a korábbi split CSV-k újrahasználhatók.


In [ ]:
export_masks_for_classification_dataset(
    model=model,
    input_root=RAW_DIR,
    output_root=LUNG_MASK_DIR,
    image_size=SEG_IMAGE_SIZE,
    mode="mask",
    threshold=0.5,
)

export_masks_for_classification_dataset(
    model=model,
    input_root=RAW_DIR,
    output_root=LUNG_MASKED_DIR,
    image_size=SEG_IMAGE_SIZE,
    mode="masked",
    threshold=0.5,
)

export_masks_for_classification_dataset(
    model=model,
    input_root=RAW_DIR,
    output_root=LUNG_CROP_DIR,
    image_size=SEG_IMAGE_SIZE,
    mode="crop",
    threshold=0.5,
)

export_summary = {
    "input_root": str(RAW_DIR),
    "lung_mask_dir": str(LUNG_MASK_DIR),
    "lung_masked_dir": str(LUNG_MASKED_DIR),
    "lung_crop_dir": str(LUNG_CROP_DIR),
    "segmentation_model_dir": str(seg_model_dir),
    "segmentation_image_size": SEG_IMAGE_SIZE,
    "threshold": 0.5,
}
save_json(export_summary, SEGMENTATION_DIR / "export_summary.json")
export_summary

In [ ]:
from PIL import Image

def _list_images(root):
    root = Path(root)
    return sorted([
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}
    ])

raw_files = _list_images(RAW_DIR)
masked_files = _list_images(LUNG_MASKED_DIR)
crop_files = _list_images(LUNG_CROP_DIR)

print("raw    :", len(raw_files))
print("masked :", len(masked_files))
print("crop   :", len(crop_files))

n = min(4, len(raw_files))
if n > 0:
    chosen = random.sample(raw_files, n)
    fig, axes = plt.subplots(n, 3, figsize=(10, 3 * n))
    if n == 1:
        axes = np.array([axes])

    for i, raw_path in enumerate(chosen):
        rel = raw_path.relative_to(RAW_DIR)
        masked_path = LUNG_MASKED_DIR / rel
        crop_path = LUNG_CROP_DIR / rel

        imgs = [
            np.array(Image.open(raw_path).convert("L")),
            np.array(Image.open(masked_path).convert("L")) if masked_path.exists() else np.zeros((32, 32)),
            np.array(Image.open(crop_path).convert("L")) if crop_path.exists() else np.zeros((32, 32)),
        ]
        titles = ["raw", "lung_masked", "lung_crop"]

        for j, (img, title) in enumerate(zip(imgs, titles)):
            axes[i, j].imshow(img, cmap="gray")
            axes[i, j].set_title(f"{title}\n{rel.name}")
            axes[i, j].axis("off")

    plt.tight_layout()
    plt.show()

## Következő lépés

A következő notebookban az **EfficientNetB0** fog tanulni:

- `data_variant="raw"`
- `data_variant="lung_masked"`
- `data_variant="lung_crop"`

és a cél az lesz, hogy összehasonlítsuk:

- a korábbi nyers képes EfficientNet eredményt
- az új, szegmentációval előfeldolgozott képeken tanult EfficientNet eredményeket


In [ ]:
try:
    from google.colab import _message
    _message.blocking_request("request_save", timeout_sec=10)
    print("Notebook save requested.")
except Exception as e:
    print("Colab save skipped:", e)